In [61]:
import requests
import pytz
import pandas as pd
import redis
import yaml
from os.path import abspath, isfile
from market import TradisAdapter
from datetime import datetime, timezone, timedelta
from io import StringIO


pd.options.display.width = 300
pd.options.display.max_rows = 1500
pd.options.display.max_columns = None
pd.options.display.max_colwidth = None
pd.options.display.expand_frame_repr = False


chicago_tz = pytz.timezone('America/Chicago')


# Функции загрузки данных должны 
# возвращать Dataframe с одинаковым 
# количеством колонок в одинаковом порядке


DATA_BASE_DIR = abspath('..') + "/data"


def get_file_name(exchange, symbol, data_type, date):
    data_type = data_type.replace("_", "")
    return f"{DATA_BASE_DIR}/{exchange}/{symbol}/{data_type}/{date:%Y-%m-%d}.txt"


def get_splits_file_name(exchange, symbol):
    return f"{DATA_BASE_DIR}/{exchange}/{symbol}/splits.txt"


def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days) + 1):
        yield start_date + timedelta(n)


def get_ib_file(ticker, start=None, end=None):
    """
    Загрузка исторических данных из файловой системы.
    """
    symbol, exchange = ticker.split(".")

    TRADES_NUM_COL = ["open", "high", "low", "close", "volume", "average", "barCount"]
    data_type = "TRADES"

    # Загрузить нужные дни
    data = ""
    for date in daterange(start - timedelta(days=1), end):
        path = get_file_name(exchange, symbol, data_type, date)
        if isfile(path):
            data += open(path).read()
        else:
            print(f"No file: {path}")

    if not data:
        return

    df = pd.read_csv(StringIO(data), sep="\t", index_col="date", dtype=str)
    df = df[df["rth"] != "rth"]  # убрать заголовочные строки
    df.index = pd.to_datetime(df.index, utc=False)
    df.sort_index(inplace=True)

    # Отфильтровать данные по времени
    df = df.loc[start:end]

    # Добавляю фейковые записи в минутных промежутках (только в основную сессию).
    # OHLC и average равны последнему известному close.
    # if data_type == "TRADES":
    #     df1 = df.resample('1T').ffill()

    #     df1["volume"] = df["volume"]
    #     df1["volume"].fillna("0", inplace=True)

    #     df1["barCount"] = df["barCount"]
    #     df1["barCount"].fillna("0", inplace=True)

    #     df1.loc[df1['volume'] == "0", ["open", "high", "low", "average"]] = df1["close"]

    #     df1 = df1[(df1["barCount"] != "0") | (df1["rth"] == "1")]
    #     df = df1

    df[TRADES_NUM_COL] = df[TRADES_NUM_COL].apply(pd.to_numeric)
    
    df = df[["open", "high", "low", "close", "volume"]]

    df["volume"] = df["volume"] * 100

    return df


def get_paraquet():
    """
    Выгрузка в формате Parquet, которую выкладывали в твиттере.
    """
    df = pd.read_parquet("../jup/URA.parquet")

    df = df.set_index("datetime")
    df.columns = ["o", "h", "l", "c", "v"]

    df = df.astype("float64").round(2)

    return df


def get_polygon(symbol, dt1, dt2):
    """
    Данные из файла, сохраненного из Polygon.
    """

    symbol = symbol.split(".")[0]
    f_name = f"../data/polygon_nyse/pre_2022/{symbol}.csv"
    df = pd.read_csv(open(f_name))
    df["dt"] = pd.to_datetime(df["t"], unit="s")

    # df["dt"] = df["dt"].dt.tz_localize("UTC").dt.tz_convert("US/Eastern").dt.tz_localize(None)

    df = df.set_index("dt")
    df = df[["o", "h", "l", "c", "v"]]

    df = df.astype("float64").round(4)
    df["v"] = df["v"].astype("int64")

    return df[dt1:dt2]


def get_tradis(symbol, dt1, dt2):
    """
    Загрузка из tradis (по адресу из конфига).
    """

    config = yaml.full_load(open(abspath("../config/bot.yaml")))

    redis_config = config["sources"]["redis_cloud"]
    redis_client = redis.Redis(**redis_config)

    class FakeSchedule:
        def is_rth(self, *args, **kwargs):
            return True

    tradis = TradisAdapter(redis_client)
    tradis.schedule = FakeSchedule()

    res = [ln[2] for ln in tradis.load([symbol], dt1, dt2)]
    df = pd.DataFrame(res)
    df = df.set_index("dt")
    df = df[["o", "h", "l", "c", "vol"]]
    df = df.dropna()
    df["vol"] = df["vol"] * 100

    df = df[df["vol"] > 0]

    return df[dt1:dt2]
 

def get_ib_tws(symbol, dt1, dt2):
    """
    Загрузка из TWS онлайн.
    """
    from ib_insync import IB, Stock, util

    util.startLoop()
    pd.options.display.width = 100
    clientId = 2
    port = 4001

    ticker, exchange = symbol.split(".")

    with IB() as ib:
        
        ib.connect(port=port, clientId=clientId)

        contract = Stock(ticker, primaryExchange=exchange, exchange="SMART")
        ib.qualifyContracts(contract)
        
        print(contract)

        res = ib.reqHistoricalData(
            contract, 
            endDateTime="", 
            durationStr="5 D", 
            barSizeSetting="1 min", 
            whatToShow="TRADES", 
            useRTH=False
        )

        df = util.df(res)

        df["date"] = df["date"].dt.tz_localize("US/Pacific").dt.tz_convert("UTC").dt.tz_localize(None)
        df = df.set_index("date")

        df = df[dt1:dt2]

        df = df[["open", "high", "low", "close", "volume"]]

        df = df[df["volume"] > 0]

        return df


def compare_columns(row):
    """
    Попарно сравнивает колонки.
    """
    lv = list(row.values)
    n = len(lv) // 2
    style = "background: #dac"
    styles = ["" if abs(lv[i] - lv[i+n]) <= 0 else style for i in range(n)]
    return styles * 2


dt1 = datetime(2018, 1, 3)
dt2 = dt1 + timedelta(hours=15)

symbol = "URA.ARCA"

print("date range:", dt1, dt2)
print("symbol:", symbol)

# df1 = get_paraquet()[dt1:dt2]
df2 = get_polygon(symbol, dt1, dt2)
# df3 = get_tradis(symbol, dt1, dt2)
# df3 = get_ib_tws(symbol, dt1, dt2)
df3 = get_ib_file(symbol, dt1, dt2)

df = pd.concat([df2, df3], axis=1).round(2)

df.columns = list(range(len(df.columns)))
res = df.style.apply(compare_columns, axis=1).format(precision=2)
res





date range: 2018-01-03 00:00:00 2018-01-03 15:00:00
symbol: URA.ARCA


,0,1,2,3,4,5,6,7,8,9
2018-01-03 14:30:00,15.80,15.91,15.75,15.79,61938,15.80,15.91,15.74,15.79,44600
2018-01-03 14:31:00,15.78,15.78,15.70,15.70,4624,15.78,15.78,15.70,15.70,4500
2018-01-03 14:32:00,15.77,15.77,15.77,15.77,100,15.77,15.77,15.76,15.77,100
2018-01-03 14:33:00,15.75,15.79,15.73,15.74,3600,15.75,15.79,15.72,15.75,3500
2018-01-03 14:34:00,15.78,15.79,15.73,15.74,780,15.78,15.79,15.73,15.74,700
2018-01-03 14:35:00,15.75,15.77,15.74,15.74,4322,15.75,15.77,15.74,15.74,4000
2018-01-03 14:36:00,15.73,15.73,15.65,15.65,14700,15.73,15.73,15.65,15.65,14300
2018-01-03 14:37:00,15.66,15.66,15.64,15.64,2830,15.66,15.66,15.63,15.64,2800
2018-01-03 14:38:00,15.65,15.66,15.65,15.65,10100,15.65,15.66,15.65,15.65,10100
2018-01-03 14:39:00,15.65,15.65,15.64,15.64,15200,15.65,15.65,15.64,15.64,15100
